In [1]:
import pandas as pd


In [2]:
learn_df = pd.read_csv("modal_experiment_v1_training_norming_candidates_with_comprehension_classified.csv"
                       ).drop(columns=["index", "index.1"])

# throw out wh-question and individual scenarios that performed worst during norming
to_keep = ["T3","T5","T7","T9","T11","T12"]
learn_df = learn_df[learn_df["item_id"].isin(to_keep)].reset_index(drop=True)

# change instances of 'mip' to 'dax'
learn_df["lexical_adjective_target"] = learn_df["lexical_adjective_target"].apply(
    lambda x: x.replace("mip", "dax")
)

learn_df.head()


,phase,item_id,domain,training_flavor,syntax,D_status,E_status,temporal_orientation,order_pair_id,order_1,...,expected_context_D,expected_context_E,expected_whole_scenario_deontic_interpretation,expected_whole_scenario_epistemic_interpretation,notes,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct,comprehension_type
0,Training,T3,Community center,Root/deontic,Declarative,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,D+,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Positive-only root training candidate. The con...,Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,A,diagnostic_strong
1,Training,T3,Community center,Epistemic,Declarative,D unspecified,E+ (weak / indirect evidence),past perfect / present progressive or stative,TE,N-E,...,D should be 'The story does not say',E+,Expected non-deontic interpretation,Expected epistemic interpretation,Positive-only epistemic training candidate. We...,Which area has the front desk ruled out?,The fitness area,The swimming area,A,diagnostic_strong
2,Training,T5,Parking,Root/deontic,Yes/no inversion,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,D+,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Positive-only root training candidate. The con...,What kind of event is Omar attending?,A museum exhibition,A concert,B,neutral
3,Training,T5,Parking,Epistemic,Yes/no inversion,D unspecified,E+ (weak / indirect evidence),past perfect / present progressive or stative,TE,N-E,...,D should be 'The story does not say',E+,Expected non-deontic interpretation,Expected epistemic interpretation,Positive-only epistemic training candidate. We...,What kind of event did Omar attend?,A museum exhibition,A concert,B,neutral
4,Training,T7,Art studio,Root/deontic,Declarative,D+,E unspecified / not ruled out,future eventive,TR,N-D,...,D+,E should not be ruled out,Expected root/deontic interpretation,Expected non-epistemic interpretation,Positive-only root training candidate. The con...,What enables Eli to use the print room?,His advanced-course qualification,A visitor wristband,A,diagnostic_weak


In [3]:
learn_df["comprehension_keep"] = (learn_df["comprehension_type"].isin(["diagnostic_strong", "diagnostic_moderate", "diagnostic_weak"]))
print(learn_df["comprehension_keep"].tolist())

learn_df["comprehension_question"] = learn_df.apply(
    lambda x: x["comprehension_question"] if x["comprehension_keep"] else "",
    axis=1
)
learn_df["comprehension_question"].to_list()


[True, True, False, False, True, True, True, True, False, False, False, False]


["Which area does Nora's membership allow her to use?",
 'Which area has the front desk ruled out?',
 '',
 '',
 'What enables Eli to use the print room?',
 'What had recently been used in the studio?',
 'What does Mateo have for entering the festival?',
 "Which gate does Mateo's friend know he did not use?",
 '',
 '',
 '',
 '']

In [4]:
print(learn_df["E_status"].value_counts())

print(learn_df["D_status"].value_counts())


E_status
E unspecified / not ruled out    6
E+ (weak / indirect evidence)    6
Name: count, dtype: int64
D_status
D+               6
D unspecified    6
Name: count, dtype: int64


In [5]:
learn_df["D_status"] = learn_df["D_status"].map({"D+": 1, "D unspecified": 0, "D-": -1})
learn_df["E_status"] = learn_df["E_status"].map({"E+ (weak / indirect evidence)": 1, "E unspecified / not ruled out": 0})

learn_df["E_status"].tolist()


[0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]

In [6]:
order1_df = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_1",
            "context_order_1",
            "functional_auxiliary_target",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "order_1":"order",
                "context_order_1":"context_order"
                })
order1_df["order_group"] = (order1_df.reset_index().index % 4) < 2

order2_df = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_2",
            "context_order_2",
            "functional_auxiliary_target",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"]].rename(columns={
                "order_2":"order",
                "context_order_2":"context_order"
                })
order2_df["order_group"] = (order1_df.reset_index().index % 4) >= 2
learn_df = pd.concat([order1_df, order2_df])
learn_df["order_group"] = learn_df["order_group"].apply(int).apply(str)

print(order1_df["order_group"].tolist())
print(order2_df["order_group"].tolist())

print(learn_df["order"].tolist())
learn_df.groupby(["order_group", "training_flavor"])["order"].value_counts()


[True, True, False, False, True, True, False, False, True, True, False, False]
[False, False, True, True, False, False, True, True, False, False, True, True]
['N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'N-D', 'N-E', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N', 'D-N', 'E-N']


order_group  training_flavor  order
0            Epistemic        E-N      3
                              N-E      3
             Root/deontic     D-N      3
                              N-D      3
1            Epistemic        E-N      3
                              N-E      3
             Root/deontic     D-N      3
                              N-D      3
Name: count, dtype: int64

In [7]:

functional = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_group",
            "order",
            "context_order",
            "functional_auxiliary_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "functional_auxiliary_target":"alien_target_response",
                })
functional["modal_type"] = "functional"

lexical = learn_df[[
            "phase",
            "item_id",
            "domain",
            "training_flavor",
            "D_status",
            "E_status",
            "temporal_orientation",
            "order_group",
            "order",
            "context_order",
            "lexical_adjective_target",
            "comprehension_question",
            "comprehension_option_A",
            "comprehension_option_B",
            "comprehension_correct"
            ]].rename(columns={
                "lexical_adjective_target":"alien_target_response",
                })
lexical["modal_type"] = "lexical"

learn_df = pd.concat([functional, lexical])
learn_df.head()


,phase,item_id,domain,training_flavor,D_status,E_status,temporal_orientation,order_group,order,context_order,alien_target_response,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct,modal_type
0,Training,T3,Community center,Root/deontic,1,0,future eventive,1,N-D,Nora is going to the community center this aft...,"Oh, so nora dax use the pool this afternoon.",Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,A,functional
1,Training,T3,Community center,Epistemic,0,1,past perfect / present progressive or stative,1,N-E,Nora is at the community center right now. The...,"Oh, so nora dax be swimming now.",Which area has the front desk ruled out?,The fitness area,The swimming area,A,functional
2,Training,T5,Parking,Root/deontic,1,0,future eventive,0,N-D,Omar is driving to a concert tomorrow. Omar's ...,"Oh, so dax Omar park in the west lot tomorrow?...",,A museum exhibition,A concert,B,functional
3,Training,T5,Parking,Epistemic,0,1,past perfect / present progressive or stative,0,N-E,Omar drove to a concert earlier today. The con...,"Oh, so dax Omar be parked in the west lot now?...",,A museum exhibition,A concert,B,functional
4,Training,T7,Art studio,Root/deontic,1,0,future eventive,1,N-D,Eli is taking an art workshop tomorrow. Eli's ...,"Oh, so eli dax use the print room tomorrow.",What enables Eli to use the print room?,His advanced-course qualification,A visitor wristband,A,functional


In [8]:
def fix_name(x):
    s = x.split(" ")
    name = s[2]
    if name == "it" or name == "is" or name == "dax":
        return x
    print(name)
    s[2] = name.capitalize()
    x = " ".join(s)
    print(x)
    return x

# fix uncapitalized names
learn_df["alien_target_response"] = learn_df["alien_target_response"].apply(fix_name)


nora
Oh, so Nora dax use the pool this afternoon.
nora
Oh, so Nora dax be swimming now.
eli
Oh, so Eli dax use the print room tomorrow.
eli
Oh, so Eli dax have worked in the print room this morning.
nina
Oh, so Nina dax work at the cooking station tomorrow.
nina
Oh, so Nina dax be working at the cooking station now.
daniel
Oh, so Daniel dax work in Lab B tomorrow.
daniel
Oh, so Daniel dax be in Lab B now.
nora
Oh, so Nora dax use the pool this afternoon.
nora
Oh, so Nora dax be swimming now.
eli
Oh, so Eli dax use the print room tomorrow.
eli
Oh, so Eli dax have worked in the print room this morning.
nina
Oh, so Nina dax work at the cooking station tomorrow.
nina
Oh, so Nina dax be working at the cooking station now.
daniel
Oh, so Daniel dax work in Lab B tomorrow.
daniel
Oh, so Daniel dax be in Lab B now.


In [9]:
learn_df["comprehension_correct"] = learn_df.apply(
    lambda x: x["comprehension_option_A"] if x["comprehension_correct"] == "A" else x["comprehension_option_B"],
    axis=1
)

learn_df["question_response"] = learn_df["alien_target_response"].apply(
    lambda x: x.split(" — ")[1] if " — " in x else ""
)
learn_df["alien_target_response"] = learn_df["alien_target_response"].apply(
    lambda x: x.split(" — ")[0]
)
# make dax bolded
learn_df["alien_target_response"] = learn_df["alien_target_response"].apply(
    lambda x: x.replace("dax", "<b>dax</b>")
)

learn_df["group"] = learn_df.apply(
    lambda x: "D->E_" + x["modal_type"] if x["training_flavor"] == "Root/deontic" else "E->D_" + x["modal_type"],
    axis=1
) + "_" + learn_df["order_group"]


learn_df.to_csv("exp1_learning_stimulus.csv")
learn_df.head()


,phase,item_id,domain,training_flavor,D_status,E_status,temporal_orientation,order_group,order,context_order,alien_target_response,comprehension_question,comprehension_option_A,comprehension_option_B,comprehension_correct,modal_type,question_response,group
0,Training,T3,Community center,Root/deontic,1,0,future eventive,1,N-D,Nora is going to the community center this aft...,"Oh, so Nora <b>dax</b> use the pool this after...",Which area does Nora's membership allow her to...,The swimming pool,The faculty practice room,The swimming pool,functional,,D->E_functional_1
1,Training,T3,Community center,Epistemic,0,1,past perfect / present progressive or stative,1,N-E,Nora is at the community center right now. The...,"Oh, so Nora <b>dax</b> be swimming now.",Which area has the front desk ruled out?,The fitness area,The swimming area,The fitness area,functional,,E->D_functional_1
2,Training,T5,Parking,Root/deontic,1,0,future eventive,0,N-D,Omar is driving to a concert tomorrow. Omar's ...,"Oh, so <b>dax</b> Omar park in the west lot to...",,A museum exhibition,A concert,A concert,functional,Yes.,D->E_functional_0
3,Training,T5,Parking,Epistemic,0,1,past perfect / present progressive or stative,0,N-E,Omar drove to a concert earlier today. The con...,"Oh, so <b>dax</b> Omar be parked in the west l...",,A museum exhibition,A concert,A concert,functional,Yes.,E->D_functional_0
4,Training,T7,Art studio,Root/deontic,1,0,future eventive,1,N-D,Eli is taking an art workshop tomorrow. Eli's ...,"Oh, so Eli <b>dax</b> use the print room tomor...",What enables Eli to use the print room?,His advanced-course qualification,A visitor wristband,His advanced-course qualification,functional,,D->E_functional_1


In [10]:
learn_df.groupby(["group", "training_flavor", "modal_type"])["order"].value_counts()


group              training_flavor  modal_type  order
D->E_functional_0  Root/deontic     functional  D-N      3
                                                N-D      3
D->E_functional_1  Root/deontic     functional  D-N      3
                                                N-D      3
D->E_lexical_0     Root/deontic     lexical     D-N      3
                                                N-D      3
D->E_lexical_1     Root/deontic     lexical     D-N      3
                                                N-D      3
E->D_functional_0  Epistemic        functional  E-N      3
                                                N-E      3
E->D_functional_1  Epistemic        functional  E-N      3
                                                N-E      3
E->D_lexical_0     Epistemic        lexical     E-N      3
                                                N-E      3
E->D_lexical_1     Epistemic        lexical     E-N      3
                                                N-E      3
Na

In [11]:
from pathlib import Path

# Testing condition is between participants. Learning cells above are unchanged.
# Put these source CSVs beside the notebook (or in upload/).
def find_test_source(*filenames):
    for folder in (Path("."), Path("upload"), Path("../upload")):
        for filename in filenames:
            path = folder / filename
            if path.is_file():
                return path
    raise FileNotFoundError(f"Cannot find any of {filenames}")

condition_files = {
    "flavour_positive": find_test_source(
        "modal_experiment_v1_positive_flavour_stimulus_candidates.csv",
        "modal_experiment_v1_positive_flavour_stimulus_candidates(1).csv"),
    "flavour_negative": find_test_source(
        "modal_experiment_v1_negative_flavour_stimulus_candidates.csv"),
    "force_negative": find_test_source(
        "modal_experiment_v1_negative_force_stimulus_candidates.csv"),
}

test_sources = []
for condition, path in condition_files.items():
    source = pd.read_csv(path)
    source = source[source["phase"].isin(["Filler", "Critical Test"])].copy()
    source["condition"] = condition
    test_sources.append(source)
test_df = pd.concat(test_sources, ignore_index=True)
test_df.groupby(["condition", "phase"]).size()


condition         phase        
flavour_negative  Critical Test    16
                  Filler            8
flavour_positive  Critical Test    16
                  Filler            8
force_negative    Critical Test    16
                  Filler            8
dtype: int64

In [12]:
# change instances of 'mip' to 'dax'
test_df["lexical_adjective_target"] = test_df["lexical_adjective_target"].apply(
    lambda x: x.replace("mip", "dax")
)

# Existing exclusions retained; revise these after the new norming results.
to_drop = ["F1", "F6", "C1", "C2", "C4", "C5"]
test_df = test_df[~test_df["item_id"].isin(to_drop)].reset_index(drop=True)
test_df.head()


,index,phase,item_id,domain,direction_or_condition,D_status,E_status,deontic_support_type,epistemic_support_type,temporal_orientation,...,context_only_possibility_question,context_only_necessity_question,whole_scenario_possibility_reading,whole_scenario_necessity_reading,context_only_possibility_response_options,context_only_necessity_response_options,expected_context_P,expected_context_Nec,expected_whole_scenario_possibility_interpretation,expected_whole_scenario_necessity_interpretation
0,34,Critical Test,C3,Student film studio,Root→Epistemic,D-,E+,explicit role-based restriction,open because the production record is incomplete,past perfect,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,35,Critical Test,C3,Student film studio,Epistemic→Root,D+,E-,explicit badge-based permission,categorically ruled out by an explicit future ...,future eventive,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,40,Critical Test,C6,Airport premium lounge,Root→Epistemic,D-,E+,explicit ticket-class restriction,open because location and entry records are in...,past perfect,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41,Critical Test,C6,Airport premium lounge,Epistemic→Root,D+,E-,explicit ticket-based permission,categorically ruled out by an explicit future ...,future eventive,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,42,Critical Test,C7,College radio station,Root→Epistemic,D-,E+,explicit membership-based restriction,open because Elena's current location is unknown,present progressive,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# `direction` keeps its original role: it identifies the LEARNING group.
# Flavour tests transfer to the other flavour; force tests stay in the learned
# flavour and test possibility -> necessity under negation.
def learning_direction(row):
    label = row["direction_or_condition"]
    if row["condition"] == "force_negative":
        if label.startswith("Root possibility"):
            return "D->E"  # Root-trained participants: root force test.
        if label.startswith("Epistemic possibility"):
            return "E->D"  # Epistemic-trained participants: epistemic force test.
    else:
        if label.startswith("Root→Epistemic"):
            return "D->E"
        if label.startswith("Epistemic→Root"):
            return "E->D"
    raise ValueError(f"Unrecognized critical condition: {label}")

criticals = test_df[test_df["phase"] == "Critical Test"].copy()
criticals["direction"] = criticals.apply(learning_direction, axis=1)

# Preserve the original logic: all fillers are shared by both learning groups,
# within their testing condition only.
fillers = test_df[test_df["phase"] == "Filler"].copy()
test_df = pd.concat([
    criticals,
    fillers.assign(direction="D->E"),
    fillers.assign(direction="E->D"),
], ignore_index=True)
test_df.groupby(["condition", "phase", "direction"]).size()


condition         phase          direction
flavour_negative  Critical Test  D->E         4
                                 E->D         4
                  Filler         D->E         6
                                 E->D         6
flavour_positive  Critical Test  D->E         4
                                 E->D         4
                  Filler         D->E         6
                                 E->D         6
force_negative    Critical Test  D->E         4
                                 E->D         4
                  Filler         D->E         6
                                 E->D         6
dtype: int64

In [14]:
# Keep the two order groups used by the learning phase. Each item appears
# once per group, and gets the opposite existing context version in group 1.
# Balance within each phase/pair first, then across the whole test block.
from itertools import product
from collections import Counter

# The current sources have one N-first pair per condition.
order_pair_orders = {
    "flavour_positive": {"N_FIRST": {"N-E-D", "N-D-E"}},
    "flavour_negative": {"N_FIRST": {"N-E-D", "N-D-E"}},
    "force_negative": {"N_FIRST": {"N-P-Nec", "N-Nec-P"}},
}
base_test = test_df.reset_index(drop=True).copy()
assert not base_test.duplicated(["condition", "direction", "item_id"]).any()
for row in base_test.itertuples(index=False):
    assert {row.order_1, row.order_2} == order_pair_orders[row.condition][row.order_pair_id]

order_groups = []
for (condition, direction), block in base_test.groupby(["condition", "direction"], sort=True):
    all_test_orders = sorted(set().union(*order_pair_orders[condition].values()))
    block = block.sort_values(["phase", "item_id"]).reset_index(drop=True)
    strata = list(block.groupby(["phase", "order_pair_id"]).indices.values())
    records = block.to_dict("records")
    best_score, best_choices = None, None
    # With the current exclusions: 12 items per block (4096 candidates).
    # This uses the actual
    # available orders rather than assuming the row index encodes the pair.
    for choices in product([0, 1], repeat=len(block)):
        if any(abs(sum(choices[i] for i in indices) * 2 - len(indices)) > 1
               for indices in strata):
            continue
        counts_by_group = []
        for group_code in [0, 1]:
            counts = Counter(
                row[f"order_{1 + (choices[i] ^ group_code)}"]
                for i, row in enumerate(records)
            )
            counts_by_group.append(counts)
        # Minimize the worst group's spread, then squared order counts.
        # Finally prefer an even split of first/second versions per phase.
        spread = max(max(c.get(o, 0) for o in all_test_orders) -
                     min(c.get(o, 0) for o in all_test_orders)
                     for c in counts_by_group)
        squared_counts = sum(c.get(o, 0) ** 2
                             for c in counts_by_group for o in all_test_orders)
        phase_split = sum(
            (sum(choices[i] for i in indices) * 2 - len(indices)) ** 2
            for indices in block.groupby("phase").indices.values()
        )
        score = (spread, squared_counts, phase_split)
        if best_score is None or score < best_score:
            best_score, best_choices = score, choices
    assert best_choices is not None, "No balanced assignment found"

    for group_code in [0, 1]:
        selected = block.copy()
        selected["order_group"] = str(group_code)
        selected["order_version"] = [1 + (choice ^ group_code) for choice in best_choices]
        selected["order"] = [row[f"order_{version}"] for row, version in
                             zip(records, selected["order_version"])]
        selected["context_order"] = [row[f"context_order_{version}"] for row, version in
                                     zip(records, selected["order_version"])]
        order_groups.append(selected)

test_df = pd.concat(order_groups, ignore_index=True)
assert test_df.groupby(["condition", "direction", "item_id", "order_group"]).size().eq(1).all()
assert test_df.groupby(["condition", "direction", "item_id"])["order_version"].nunique().eq(2).all()

# Inspect available pairs and the resulting counts for each condition.
pair_counts = base_test.groupby(["condition", "direction", "phase", "order_pair_id"]).size()
print("Available pair counts:\n", pair_counts)
test_df.groupby(["condition", "order_group", "direction", "phase"])["order"].value_counts()


Available pair counts:
 condition         direction  phase          order_pair_id
flavour_negative  D->E       Critical Test  N_FIRST          4
                             Filler         N_FIRST          6
                  E->D       Critical Test  N_FIRST          4
                             Filler         N_FIRST          6
flavour_positive  D->E       Critical Test  N_FIRST          4
                             Filler         N_FIRST          6
                  E->D       Critical Test  N_FIRST          4
                             Filler         N_FIRST          6
force_negative    D->E       Critical Test  N_FIRST          4
                             Filler         N_FIRST          6
                  E->D       Critical Test  N_FIRST          4
                             Filler         N_FIRST          6
dtype: int64


condition         order_group  direction  phase          order  
flavour_negative  0            D->E       Critical Test  N-D-E      2
                                                         N-E-D      2
                                          Filler         N-D-E      3
                                                         N-E-D      3
                               E->D       Critical Test  N-D-E      2
                                                         N-E-D      2
                                          Filler         N-D-E      3
                                                         N-E-D      3
                  1            D->E       Critical Test  N-D-E      2
                                                         N-E-D      2
                                          Filler         N-D-E      3
                                                         N-E-D      3
                               E->D       Critical Test  N-D-E      2
                         

In [15]:
# Keep flavour and force statuses separate. Inapplicable cells remain missing.
status_maps = {
    "D_status": {"D+": 1, "D unspecified": 0, "D-": -1},
    "E_status": {"E+": 1, "E-": -1},
    "P_status": {"P+": 1, "P-": -1},
    "Nec_status": {"Nec+": 1, "Nec-": -1},
}
for column, mapping in status_maps.items():
    values = test_df[column]
    assert values.dropna().isin(mapping).all(), f"Unexpected {column} label"
    test_df[column] = values.map(mapping).astype("Int64")

# Expected endorsement of the displayed target under successful transfer.
# Accept/Reject are semantic labels, independent of button positions or coding.
# Criticals: evaluate the transferred flavour, or necessity for force tests.
# Fillers: both relevant readings must agree, so transfer is not needed.
def expected_test_choice(row):
    if row["condition"] == "force_negative":
        status = row["Nec_status"]
        if row["phase"] == "Filler":
            assert row["P_status"] == status, "Force filler readings disagree"
    else:
        target_status = "E_status" if row["direction"] == "D->E" else "D_status"
        status = row[target_status]
        if row["phase"] == "Filler":
            assert row["D_status"] == row["E_status"], "Flavour filler readings disagree"
    assert pd.notna(status) and status in (-1, 1), "Expected choice needs a definite status"
    target_true = status == 1
    if row["condition"] in ("flavour_negative", "force_negative"):
        target_true = not target_true  # Outer negation reverses endorsement.
    return "Accept" if target_true else "Reject"

test_df["expected_choice"] = test_df.apply(expected_test_choice, axis=1)
assert test_df.loc[test_df["phase"] == "Critical Test", "expected_choice"].eq("Accept").all()

common_columns = [
    "phase", "item_id", "domain", "condition", "direction",
    "direction_or_condition", "D_status", "E_status", "P_status", "Nec_status",
    "temporal_orientation", "order_group", "order", "context_order",
    "expected_choice",
]
modal_versions = []
for modal_type, target_column in [
    ("functional", "functional_auxiliary_target"),
    ("lexical", "lexical_adjective_target"),
]:
    version = test_df[common_columns + [target_column]].rename(
        columns={target_column: "alien_target_response"}).copy()
    version["modal_type"] = modal_type
    modal_versions.append(version)
test_df = pd.concat(modal_versions, ignore_index=True)

# Learning CSV stays unchanged. Assign a participant ONE test `group`, then
# select learning rows using its `learning_group` (the original group name).
test_df["learning_group"] = (
    test_df["direction"] + "_" + test_df["modal_type"] + "_" + test_df["order_group"])
test_df["group"] = test_df["learning_group"] + "_" + test_df["condition"]

# make 'dax' bolded
test_df["alien_target_response"] = test_df["alien_target_response"].apply(
    lambda x: x.replace("dax", "<b>dax</b>")
)
test_df.head()


,phase,item_id,domain,condition,direction,direction_or_condition,D_status,E_status,P_status,Nec_status,temporal_orientation,order_group,order,context_order,expected_choice,alien_target_response,modal_type,learning_group,group
0,Critical Test,C3,Student film studio,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-E-D,Grace is returning to the film studio tomorrow...,Accept,"Oh, so it is not the case that Grace <b>dax</b...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
1,Critical Test,C6,Airport premium lounge,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-E-D,Marcus is flying from the airport again tomorr...,Accept,"Oh, so it is not the case that Marcus <b>dax</...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
2,Critical Test,C7,College radio station,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-D-E,Elena starts an internship at the radio statio...,Accept,"Oh, so it is not the case that Elena <b>dax</b...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
3,Critical Test,C8,Ski lodge equipment storage,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-D-E,Ravi is volunteering at a ski race tomorrow. H...,Accept,"Oh, so it is not the case that Ravi <b>dax</b>...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
4,Filler,F2,Cinema projection booth,flavour_negative,D->E,D-E- filler (negated),-1,-1,<NA>,<NA>,past perfect,0,N-E-D,Amir attended a film festival yesterday. Secur...,Accept,"Oh, so it is not the case that Amir <b>dax</b>...",functional,D->E_functional_0,D->E_functional_0_flavour_negative


In [16]:
# One testing condition per group, with a matching unchanged learning group.
assert test_df.groupby(["group", "item_id"]).size().eq(1).all()
assert test_df.groupby("group")["condition"].nunique().eq(1).all()
assert test_df.groupby("group")["learning_group"].nunique().eq(1).all()
assert set(test_df["learning_group"]) == set(learn_df["group"])
assert test_df.groupby("learning_group")["condition"].nunique().eq(3).all()
assert test_df["context_order"].notna().all()
assert test_df["alien_target_response"].notna().all()

# Within every condition, phase, and pair, order counts differ by at most one.
for (group, condition, phase), block in test_df.groupby(["group", "condition", "phase"]):
    for pair, orders in order_pair_orders[condition].items():
        paired = block[block["order"].isin(orders)]
        counts = paired["order"].value_counts().reindex(sorted(orders), fill_value=0)
        assert counts.max() - counts.min() <= 1

test_df.groupby(["condition", "group", "direction", "modal_type"])["order"].value_counts()


condition         group                               direction  modal_type  order  
flavour_negative  D->E_functional_0_flavour_negative  D->E       functional  N-D-E      5
                                                                             N-E-D      5
                  D->E_functional_1_flavour_negative  D->E       functional  N-D-E      5
                                                                             N-E-D      5
                  D->E_lexical_0_flavour_negative     D->E       lexical     N-D-E      5
                                                                             N-E-D      5
                  D->E_lexical_1_flavour_negative     D->E       lexical     N-D-E      5
                                                                             N-E-D      5
                  E->D_functional_0_flavour_negative  E->D       functional  N-D-E      5
                                                                             N-E-D      5
               

In [17]:
test_df[test_df["phase"] == "Critical Test"].groupby(
    ["condition", "group", "direction", "modal_type"])["order"].value_counts()


condition         group                               direction  modal_type  order  
flavour_negative  D->E_functional_0_flavour_negative  D->E       functional  N-D-E      2
                                                                             N-E-D      2
                  D->E_functional_1_flavour_negative  D->E       functional  N-D-E      2
                                                                             N-E-D      2
                  D->E_lexical_0_flavour_negative     D->E       lexical     N-D-E      2
                                                                             N-E-D      2
                  D->E_lexical_1_flavour_negative     D->E       lexical     N-D-E      2
                                                                             N-E-D      2
                  E->D_functional_0_flavour_negative  E->D       functional  N-D-E      2
                                                                             N-E-D      2
               

In [18]:
test_df[test_df["phase"] == "Filler"].groupby(
    ["condition", "group", "direction", "modal_type"])["order"].value_counts()


condition         group                               direction  modal_type  order  
flavour_negative  D->E_functional_0_flavour_negative  D->E       functional  N-D-E      3
                                                                             N-E-D      3
                  D->E_functional_1_flavour_negative  D->E       functional  N-D-E      3
                                                                             N-E-D      3
                  D->E_lexical_0_flavour_negative     D->E       lexical     N-D-E      3
                                                                             N-E-D      3
                  D->E_lexical_1_flavour_negative     D->E       lexical     N-D-E      3
                                                                             N-E-D      3
                  E->D_functional_0_flavour_negative  E->D       functional  N-D-E      3
                                                                             N-E-D      3
               

In [19]:
test_df.head()


,phase,item_id,domain,condition,direction,direction_or_condition,D_status,E_status,P_status,Nec_status,temporal_orientation,order_group,order,context_order,expected_choice,alien_target_response,modal_type,learning_group,group
0,Critical Test,C3,Student film studio,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-E-D,Grace is returning to the film studio tomorrow...,Accept,"Oh, so it is not the case that Grace <b>dax</b...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
1,Critical Test,C6,Airport premium lounge,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-E-D,Marcus is flying from the airport again tomorr...,Accept,"Oh, so it is not the case that Marcus <b>dax</...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
2,Critical Test,C7,College radio station,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-D-E,Elena starts an internship at the radio statio...,Accept,"Oh, so it is not the case that Elena <b>dax</b...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
3,Critical Test,C8,Ski lodge equipment storage,flavour_negative,D->E,Root→Epistemic (negated),1,-1,<NA>,<NA>,future eventive,0,N-D-E,Ravi is volunteering at a ski race tomorrow. H...,Accept,"Oh, so it is not the case that Ravi <b>dax</b>...",functional,D->E_functional_0,D->E_functional_0_flavour_negative
4,Filler,F2,Cinema projection booth,flavour_negative,D->E,D-E- filler (negated),-1,-1,<NA>,<NA>,past perfect,0,N-E-D,Amir attended a film festival yesterday. Secur...,Accept,"Oh, so it is not the case that Amir <b>dax</b>...",functional,D->E_functional_0,D->E_functional_0_flavour_negative


In [20]:
pilot_condition = test_df[test_df["condition"] == "flavour_positive"]
print(pilot_condition["group"].value_counts())

pilot_condition["group"] = pilot_condition["group"].apply(
    lambda x: x.split("_flavour")[0]
)

# fix uncapitalized names
pilot_condition["alien_target_response"] = pilot_condition["alien_target_response"].apply(fix_name)

pilot_condition.head()


group
D->E_functional_0_flavour_positive    10
D->E_functional_1_flavour_positive    10
E->D_functional_0_flavour_positive    10
E->D_functional_1_flavour_positive    10
D->E_lexical_0_flavour_positive       10
D->E_lexical_1_flavour_positive       10
E->D_lexical_0_flavour_positive       10
E->D_lexical_1_flavour_positive       10
Name: count, dtype: int64
grace
Oh, so Grace <b>dax</b> have gone into the sound-mixing suite.
marcus
Oh, so Marcus <b>dax</b> have waited in the premium lounge.
elena
Oh, so Elena <b>dax</b> be sitting in the live recording booth now.
ravi
Oh, so Ravi <b>dax</b> have gone into the equipment storage room.
amir
Oh, so Amir <b>dax</b> have entered the projection booth.
zoe
Oh, so Zoe <b>dax</b> be working at the decorating station now.
henry
Oh, so Henry <b>dax</b> practice in the faculty room tomorrow afternoon.
june
Oh, so June <b>dax</b> have attended the session in Conference Suite B.
leah
Oh, so Leah <b>dax</b> go into the equipment bay tomorrow morning.


/var/folders/lq/kd3wh6952vg99n5ppf4srz9w0000gn/T/ipykernel_9119/2565584352.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pilot_condition["group"] = pilot_condition["group"].apply(
/var/folders/lq/kd3wh6952vg99n5ppf4srz9w0000gn/T/ipykernel_9119/2565584352.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pilot_condition["alien_target_response"] = pilot_condition["alien_target_response"].apply(fix_name)


,phase,item_id,domain,condition,direction,direction_or_condition,D_status,E_status,P_status,Nec_status,temporal_orientation,order_group,order,context_order,expected_choice,alien_target_response,modal_type,learning_group,group
40,Critical Test,C3,Student film studio,flavour_positive,D->E,Root→Epistemic,-1,1,<NA>,<NA>,past perfect,0,N-E-D,Grace is working on a documentary at a student...,Accept,"Oh, so Grace <b>dax</b> have gone into the sou...",functional,D->E_functional_0,D->E_functional_0
41,Critical Test,C6,Airport premium lounge,flavour_positive,D->E,Root→Epistemic,-1,1,<NA>,<NA>,past perfect,0,N-E-D,Marcus waited several hours at the airport yes...,Accept,"Oh, so Marcus <b>dax</b> have waited in the pr...",functional,D->E_functional_0,D->E_functional_0
42,Critical Test,C7,College radio station,flavour_positive,D->E,Root→Epistemic,-1,1,<NA>,<NA>,present progressive,0,N-D-E,Elena is visiting a college radio station. The...,Accept,"Oh, so Elena <b>dax</b> be sitting in the live...",functional,D->E_functional_0,D->E_functional_0
43,Critical Test,C8,Ski lodge equipment storage,flavour_positive,D->E,Root→Epistemic,-1,1,<NA>,<NA>,past perfect,0,N-D-E,Ravi spent the morning at a ski lodge. The equ...,Accept,"Oh, so Ravi <b>dax</b> have gone into the equi...",functional,D->E_functional_0,D->E_functional_0
44,Filler,F2,Cinema projection booth,flavour_positive,D->E,D-E- filler,-1,-1,<NA>,<NA>,past perfect,0,N-E-D,Amir attended a film festival yesterday. Secur...,Reject,"Oh, so Amir <b>dax</b> have entered the projec...",functional,D->E_functional_0,D->E_functional_0


In [21]:
def remove_negated(x):
    if x["phase"] == "Filler":
        # return unomdified
        return x["context_order"]
    c = x["context_order"].split(". ")
    o = x["order"].split("-")
    assert len(c) == 3
    assert len(o) == 3
    remove_flavour = x["direction"][0]
    i = o.index(remove_flavour)
    assert (i == 1) or (i == 2)
    context = ". ".join(c[:i] + c[i+1:]) + "."
    return context

pilot_condition["new_context"] = pilot_condition.apply(
    remove_negated, axis=1
)

pilot_condition.head()


/var/folders/lq/kd3wh6952vg99n5ppf4srz9w0000gn/T/ipykernel_9119/2104009143.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pilot_condition["new_context"] = pilot_condition.apply(


,phase,item_id,domain,condition,direction,direction_or_condition,D_status,E_status,P_status,Nec_status,temporal_orientation,order_group,order,context_order,expected_choice,alien_target_response,modal_type,learning_group,group,new_context
40,Critical Test,C3,Student film studio,flavour_positive,D->E,Root→Epistemic,-1,1,<NA>,<NA>,past perfect,0,N-E-D,Grace is working on a documentary at a student...,Accept,"Oh, so Grace <b>dax</b> have gone into the sou...",functional,D->E_functional_0,D->E_functional_0,Grace is working on a documentary at a student...
41,Critical Test,C6,Airport premium lounge,flavour_positive,D->E,Root→Epistemic,-1,1,<NA>,<NA>,past perfect,0,N-E-D,Marcus waited several hours at the airport yes...,Accept,"Oh, so Marcus <b>dax</b> have waited in the pr...",functional,D->E_functional_0,D->E_functional_0,Marcus waited several hours at the airport yes...
42,Critical Test,C7,College radio station,flavour_positive,D->E,Root→Epistemic,-1,1,<NA>,<NA>,present progressive,0,N-D-E,Elena is visiting a college radio station. The...,Accept,"Oh, so Elena <b>dax</b> be sitting in the live...",functional,D->E_functional_0,D->E_functional_0,Elena is visiting a college radio station. The...
43,Critical Test,C8,Ski lodge equipment storage,flavour_positive,D->E,Root→Epistemic,-1,1,<NA>,<NA>,past perfect,0,N-D-E,Ravi spent the morning at a ski lodge. The equ...,Accept,"Oh, so Ravi <b>dax</b> have gone into the equi...",functional,D->E_functional_0,D->E_functional_0,Ravi spent the morning at a ski lodge. His fri...
44,Filler,F2,Cinema projection booth,flavour_positive,D->E,D-E- filler,-1,-1,<NA>,<NA>,past perfect,0,N-E-D,Amir attended a film festival yesterday. Secur...,Reject,"Oh, so Amir <b>dax</b> have entered the projec...",functional,D->E_functional_0,D->E_functional_0,Amir attended a film festival yesterday. Secur...


In [22]:
pilot_condition.to_csv("exp1_test_stimulus_pilot.csv")


In [23]:
test_df.to_csv("exp1_test_stimulus.csv")


Participant assignment: choose one `group` from `exp1_test_stimulus.csv` and show only test rows matching it. Use that group's `learning_group` to select rows from the unchanged `exp1_learning_stimulus.csv`. For example, `D->E_functional_0_force_negative` uses learning group `D->E_functional_0`. There are 24 testing groups: 3 conditions × 2 learned flavours × 2 modal types × 2 order groups. The legacy `direction` label identifies the learning group; in force-negative testing, the flavour remains the learned flavour.

The existing `to_keep` and `to_throw` lists are retained for later revision. Testing uses only Critical Test and Filler rows from the three supplied sets. Fillers remain shared across the two learning directions, as in the original code. No learning cell source has changed. The separate training CSV referenced in cell 1 is still required to run the full notebook.


`expected_choice` is `Accept` or `Reject` for the displayed target, accounting for outer negation. For criticals this assumes successful transfer (to the other flavour, or from possibility to necessity); all current criticals predict `Accept`. Filler readings agree and give the same expected choice with or without transfer. Labels describe endorsement, not button positions or numeric response codes. Expected choices are derived from the status columns, not item IDs.